# Description feature validation

This is an exploratory, single-reviewer audit, not forecasting model training. The random holdout contains 50 product descriptions sampled after excluding earlier reviewed samples. It is a small sample from the described products available at the April 2011 cutoff. The candidate rule was fixed before these held-out descriptions were reviewed.

In [1]:
from pathlib import Path
import json
import pandas as pd
from retailmind.product_features import candidate_pack_size
ROOT = Path.cwd() if (Path.cwd() / 'pyproject.toml').exists() else Path.cwd().parent
reviews = ROOT / 'docs/reviews'
labels = pd.read_csv(reviews / 'pack-random-heldout-labels.csv', dtype={'stock_code':str})
labels['extracted_size'] = labels.description.map(candidate_pack_size)
summary = json.loads((ROOT / 'reports/generated/pack-review-evaluation.json').read_text())
print(summary['random-heldout'])

{'rows': 50, 'explicit_count_rows': 11, 'correct_counts': 10, 'missed_counts': 1, 'incorrect_counts': 0, 'no_explicit_count_rows': 36, 'false_positives': 0, 'ambiguous_rows': 3, 'ambiguous_with_prediction': 0, 'reviewer': 'single assistant reviewer; not independently adjudicated', 'sample': 'simple random sample of remaining described products'}


## Random held-out review
Ambiguous descriptions are evaluated as abstentions, not negatives. These annotations are a single reviewer’s interpretation of titles, not verified package specifications.

In [2]:
display(labels[['stock_code', 'description', 'review_status', 'review_pack_size', 'extracted_size', 'review_note']].fillna('—'))

/home/bmage/projects/retailmind-ai-forecasting/.cache/tmp/ipykernel_11322/1846344782.py:1: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  display(labels[['stock_code', 'description', 'review_status', 'review_pack_size', 'extracted_size', 'review_note']].fillna('—'))


,stock_code,description,review_status,review_pack_size,extracted_size,review_note
0,16239,BALLERINA HEART DECORATION.,no_explicit_count,—,—,No explicit set/pack count in title
1,20617,FIRST CLASS PASSPORT COVER,no_explicit_count,—,—,No explicit set/pack count in title
2,20668,DISCO BALL CHRISTMAS DECORATION,no_explicit_count,—,—,No explicit set/pack count in title
3,20717,STRAWBERRY SHOPPER BAG,no_explicit_count,—,—,No explicit set/pack count in title
4,20863,BLUE PATCH PENCIL CASE PINK HEART,no_explicit_count,—,—,No explicit set/pack count in title
5,20940,SET/6 GARDEN GNOME T-LIGHT CANDLES,explicit_count,6.0,6.0,Explicit set/pack count or unambiguous piece c...
6,21365,MIRRORED WALL ART STARS,no_explicit_count,—,—,No explicit set/pack count in title
7,21430,SET/3 RED GINGHAM ROSE STORAGE BOX,explicit_count,3.0,3.0,Explicit set/pack count or unambiguous piece c...
8,21471,STRAWBERRY RAFFIA FOOD COVER,no_explicit_count,—,—,No explicit set/pack count in title
9,21613,S/12 VANILLA BOTANICAL T-LIGHTS,explicit_count,12.0,12.0,Explicit set/pack count or unambiguous piece c...


## Earlier targeted review and code-family review
The earlier sample was selected to emphasize specific patterns, so its scores do not estimate full-catalogue accuracy. The code-family sample is qualitative and does not prove manufacturer or family identity.

In [3]:
display(pd.DataFrame(summary).T)
display(pd.read_csv(reviews/'code-family-review.csv', dtype={'candidate_code_base':str}))

,rows,explicit_count_rows,correct_counts,missed_counts,incorrect_counts,no_explicit_count_rows,false_positives,ambiguous_rows,ambiguous_with_prediction,reviewer,sample
development,30,13,13,0,0,9,0,8,0,single assistant reviewer; not independently a...,"stratified by text patterns, not representative"
heldout,30,13,13,0,0,9,0,8,0,single assistant reviewer; not independently a...,"stratified by text patterns, not representative"
random-heldout,50,11,10,1,0,36,0,3,0,single assistant reviewer; not independently a...,simple random sample of remaining described pr...


,candidate_code_base,n_codes,review,notes
0,17129,3,plausible_related,Descriptions suggest related variants; manufac...
1,17164,2,plausible_related,Descriptions suggest related variants; manufac...
2,35638,2,plausible_related,Descriptions suggest related variants; manufac...
3,44092,2,uncertain,Broadly related terms but exact family or vari...
4,71101,2,uncertain,Broadly related terms but exact family or vari...
5,79303,3,plausible_related,Descriptions suggest related variants; manufac...
6,84201,2,plausible_related,Descriptions suggest related variants; manufac...
7,84520,3,plausible_related,Related napkins; 84520B and 84520b share text ...
8,84531,2,plausible_related,Descriptions suggest related variants; manufac...
9,85136,3,plausible_related,Descriptions suggest related variants; manufac...


## Findings and remaining work
- The random held-out sample includes an explicit `EIGHT PIECE SNAKE SET` that the current parser misses. The rule was not modified after seeing this held-out example.
- Numeric phrases such as `5 HOOK HANGER` or `20 DOLLY PEGS` remain ambiguous; unknown is not a single-item pack.
- This sample is small, products are from a retrospective universe, and labels were created by a single assistant reviewer. Independent annotation and a larger representative set are needed before treating the features as verified.
- No prediction model is trained or evaluated here. Feature snapshots must be rebuilt as of each forecast origin; future descriptions or prices must not leak into model input.